In [ ]:
!pip install -q evaluate scikit-learn rouge_score bert_score sacrebleu

In [ ]:
import json
import torch
import numpy as np
import evaluate
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load HuggingFace metrics
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

# Load your trained model and tokenizer (Update path to your saved model directory)
model_path = "./results/checkpoint-xxx" # Replace with your actual checkpoint path
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def truncate_to_50_words(text):
    if not isinstance(text, str): return ""
    return " ".join(text.split()[:50])

def parse_prediction(pred_str):
    """Splits the Seq2Seq output into a boolean/string label and an explanation."""
    pred_str = pred_str.strip()
    
    # Default fallback values
    label = "unknown"
    explanation = pred_str
    
    # Assuming the format "Label: True. Explanation: ..."
    if "Explanation:" in pred_str:
        parts = pred_str.split("Explanation:", 1)
        label_part = parts[0].replace("Label:", "").strip().lower()
        # Clean up punctuation from label
        label_part = label_part.replace(".", "").strip()
        
        # Map boolean-like strings to your specific target names
        if label_part in ["true", "1", "supported", "yes"]:
            label = "SUPPORTED"
        elif label_part in ["false", "0", "refuted", "no"]:
            label = "REFUTED"
            
        explanation = parts[1].strip()
        
    return label, explanation

def evaluate_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    true_labels = []
    pred_labels = []
    true_explanations = []
    pred_explanations = []
    
    print(f"Evaluating {file_path}...")
    for item in tqdm(data):
        claim = item.get("claim", "")
        markdown = item.get("enriched_markdown", "")
        title = truncate_to_50_words(item.get("title_description", ""))
        
        # Ground truth mappings
        gt_label_raw = str(item.get("label", "")).lower()
        gt_label = "SUPPORTED" if gt_label_raw in ["true", "1"] else "REFUTED"
        gt_explanation = item.get("explanation", "")
        
        # Format input exactly as done in training
        input_text = f"Claim: {claim} | Context: {markdown} | Title: {title}"
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(model.device)
        
        # Generate output
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=256, early_stopping=True)
            
        pred_str = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Parse output
        p_label, p_expl = parse_prediction(pred_str)
        
        true_labels.append(gt_label)
        pred_labels.append(p_label)
        true_explanations.append(gt_explanation)
        pred_explanations.append(p_expl)
        
    # Classification Metrics
    acc = accuracy_score(true_labels, pred_labels) * 100
    f1 = f1_score(true_labels, pred_labels, average='macro') * 100
    
    # NLG Metrics
    # Note: bertscore takes a while; you can comment it out during rapid testing
    bleu = bleu_metric.compute(predictions=pred_explanations, references=[[t] for t in true_explanations])["score"]
    rouge = rouge_metric.compute(predictions=pred_explanations, references=true_explanations)["rougeL"] * 100
    bert_s = bertscore_metric.compute(predictions=pred_explanations, references=true_explanations, lang="en")
    bert_f1 = np.mean(bert_s["f1"]) * 100
    
    return {
        "acc": acc, "f1": f1, "bleu": bleu, "rouge": rouge, "bertscore": bert_f1,
        "y_true": true_labels, "y_pred": pred_labels
    }

# Run evaluation on both test sets
res_test1 = evaluate_dataset("test_1_w_spec.json")
res_test2 = evaluate_dataset("test_2_w_spec.json")

In [ ]:
# Calculate Averages
avg_acc = (res_test1["acc"] + res_test2["acc"]) / 2
avg_bleu = (res_test1["bleu"] + res_test2["bleu"]) / 2
avg_rouge = (res_test1["rouge"] + res_test2["rouge"]) / 2
avg_bert = (res_test1["bertscore"] + res_test2["bertscore"]) / 2

# Baseline comparisons (Hardcoded from your example to match the layout)
baselines = [
    ("DePlot-DeBERTa-class", "C", 75.0, 75.0, 72.5, 72.5, 73.8, "-", "-", "-"),
    ("DePlot-FlanT5-finetune-multi", "M", 65.7, 65.7, 65.9, 65.8, 65.8, "17.3", "46.3", "91.5"),
    ("MatCha-finetune-multi", "M", 59.4, 59.1, 61.1, 60.9, 60.2, "17.1", "37.3", "67.8"),
    ("GPT4V (Zero-Shot)", "M", 73.8, 73.5, 72.0, 71.3, 72.9, "10.0", "32.3", "89.1")
]

# Baseline to compare against for the "+X.X% (above)" metric
nli_baseline_acc = 75.0 # Assuming DePlot-DeBERTa-class is the NLI baseline
acc_diff = avg_acc - nli_baseline_acc
diff_str = f"+{acc_diff:.1f}% (above)" if acc_diff >= 0 else f"{acc_diff:.1f}% (below)"

# Print Header
print(f"NLG Results -> BLEU: {avg_bleu:.2f} | ROUGE-L: {avg_rouge:.2f} | BERTScore: {avg_bert:.2f}\n")
print("="*115)
print("  CHARTCHECK EVALUATION MATRIX — Your Model Name".center(115))
print("="*115)
print(f"{'Model':>28} {'Task':>5} {'Test1_Acc':>9} {'Test1_F1':>9} {'Test2_Acc':>9} {'Test2_F1':>9} {'Avg_Acc':>8} {'BLEU':>6} {'ROUGE-L':>7} {'BERTScore':>9}")

# Print Baselines
for b in baselines:
    print(f"{b[0]:>28} {b[1]:>5} {b[2]:>9.1f} {b[3]:>9.1f} {b[4]:>9.1f} {b[5]:>9.1f} {b[6]:>8.1f} {b[7]:>6} {b[8]:>7} {b[9]:>9}")

# Print Our Model
print(f"{'Flan-T5 (ours)':>28} {'C+M':>5} {res_test1['acc']:>9.1f} {res_test1['f1']:>9.1f} {res_test2['acc']:>9.1f} {res_test2['f1']:>9.1f} {avg_acc:>8.1f} {avg_bleu:>6.1f} {avg_rouge:>7.1f} {avg_bert:>9.1f}")
print("="*115)

# Print Summary and Report
print(f"\n  Avg accuracy vs NLI-DeBERTa baseline : {diff_str}\n")
print(f"Test 1  →  Acc: {res_test1['acc']:.1f}%   Macro-F1: {res_test1['f1']:.1f}")
print(f"Test 2  →  Acc: {res_test2['acc']:.1f}%   Macro-F1: {res_test2['f1']:.1f}")
print(f"Average →  Acc: {avg_acc:.1f}%\n")

print("Test 1 — Per-class report:")
print(classification_report(res_test1["y_true"], res_test1["y_pred"], target_names=["REFUTED", "SUPPORTED"], digits=2))